# Leaf YOLO Training on Colab A100

This notebook clones the GitHub repository, mounts Google Drive, caches the dataset in Drive, prepares a one-class `leaf` dataset, trains YOLO26 candidates with an accuracy-focused fine-tune stage, evaluates on the test split, and exports ONNX/TF.js artifacts for browser deployment.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/leaf-object-detection'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/datasets').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/runs').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_ROOT}/artifacts').mkdir(parents=True, exist_ok=True)
print(f'Drive output folder: {DRIVE_ROOT}')

## 2. Clone Repo and Install Dependencies

In [ ]:
%cd /content
!rm -rf leaf-object-detection
!git clone https://github.com/fishman7337/leaf-object-detection.git
%cd /content/leaf-object-detection
!pip install -U -r requirements-colab.txt

## 3. Locate or Download Dataset

This cell uses fast sources only: prepared `leaf_yolo_dataset.tar.gz`, cached `archive.zip`, or automatic KaggleHub download to local Colab disk. It does not prompt for uploads.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import kagglehub

drive_archive = Path(DRIVE_ROOT) / 'archive.zip'
drive_prepared = Path(DRIVE_ROOT) / 'datasets' / 'leaf_yolo_dataset.tar.gz'
drive_raw_root = Path(DRIVE_ROOT) / 'datasets' / 'archive'
drive_raw_dataset = drive_raw_root / 'PlantVillage_for_object_detection' / 'Dataset'
drive_datasets_folder_url = 'https://drive.google.com/drive/folders/1YpqsQakKmwlgUGjYTygBM1ZIS6w-nQiI'
local_archive = Path('/content/leaf-object-detection/archive.zip')
local_kaggle_root = Path('/content/leaf-object-detection/kaggle_raw')
prepared_ready = drive_prepared.exists()
drive_raw_ready = drive_raw_dataset.exists()
local_raw_dataset = None

if prepared_ready:
    print(f'Prepared dataset already exists in Drive: {drive_prepared}')
    print('Fast path enabled: prep will be skipped.')
elif local_archive.exists():
    print(f'Local archive already exists: {local_archive}')
    if not drive_archive.exists():
        shutil.copy2(local_archive, drive_archive)
        print(f'Cached local archive to Drive: {drive_archive}')
elif drive_archive.exists():
    shutil.copy2(drive_archive, local_archive)
    print(f'Copied cached dataset archive from Drive: {drive_archive}')
else:
    print('No fast dataset file found in Drive. Downloading public dataset automatically with KaggleHub...')
    kaggle_path = Path(kagglehub.dataset_download('sebastianpalaciob/plantvillage-for-object-detection-yolo'))
    print(f'KaggleHub dataset path: {kaggle_path}')
    candidates = [p for p in [kaggle_path, *kaggle_path.rglob('Dataset')] if (p / 'images').exists() and (p / 'labels').exists()]
    if candidates:
        local_raw_dataset = candidates[0]
        print(f'Found local Kaggle raw dataset: {local_raw_dataset}')
    else:
        zip_candidates = list(kaggle_path.rglob('*.zip'))
        if not zip_candidates:
            raise FileNotFoundError(f'KaggleHub download did not contain a YOLO Dataset folder or zip file under {kaggle_path}')
        shutil.copy2(zip_candidates[0], local_archive)
        print(f'Copied Kaggle zip to local archive: {local_archive}')
    if local_archive.exists() and not drive_archive.exists():
        shutil.copy2(local_archive, drive_archive)
        print(f'Cached archive.zip to Drive for future runs: {drive_archive}')

if local_archive.exists():
    print(f'Local archive ready: {local_archive} ({local_archive.stat().st_size / 1e9:.2f} GB)')
elif prepared_ready:
    print('No local archive needed because the prepared dataset is available in Drive.')
elif local_raw_dataset is not None:
    print(f'Local raw dataset ready: {local_raw_dataset}')
elif drive_raw_ready:
    print('Loose Drive folder exists, but it is intentionally skipped because it is too slow.')

## 4. Add Mixed-Scene Public Data and Hard Negatives

In [ ]:
!mkdir -p public
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset public/PlantDoc-Object-Detection-Dataset || true
!python scripts/import_plantdoc_git.py --repo public/PlantDoc-Object-Detection-Dataset --out public/plantdoc_yolo --force
!python scripts/generate_synthetic_negatives.py --out hard_negatives/synthetic --count 300 --size 320

## 5. Restore or Prepare Dataset, Then Upload Prepared Dataset to Drive

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

drive_prepared = Path(DRIVE_ROOT) / 'datasets' / 'leaf_yolo_dataset.tar.gz'
drive_raw_root = Path(DRIVE_ROOT) / 'datasets' / 'archive'
drive_raw_dataset = drive_raw_root / 'PlantVillage_for_object_detection' / 'Dataset'
local_dataset = Path('datasets/leaf_yolo')
local_archive = Path('archive.zip')
local_raw_dataset = globals().get('local_raw_dataset')
allow_slow_drive_folder = os.environ.get('ALLOW_SLOW_DRIVE_FOLDER_PREP') == '1'

def run_checked(cmd):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
    return result

def count_files(path, suffixes=None):
    if not path.exists():
        return 0
    suffixes = {s.lower() for s in suffixes} if suffixes else None
    return sum(1 for p in path.iterdir() if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes))

if drive_prepared.exists():
    print(f'Restoring prepared dataset from Drive: {drive_prepared}')
    if local_dataset.exists():
        shutil.rmtree(local_dataset)
    run_checked(['tar', '-xzf', str(drive_prepared)])
elif local_raw_dataset is not None and Path(local_raw_dataset).exists():
    local_raw_dataset = Path(local_raw_dataset)
    print(f'Preparing from local KaggleHub dataset: {local_raw_dataset}')
    run_checked([
        'python', 'scripts/prepare_leaf_dataset.py',
        '--work-dir', '.',
        '--raw-dir', str(local_raw_dataset),
        '--no-extract',
        '--extra-yolo-dir', 'public/plantdoc_yolo',
        '--hard-negatives-dir', 'hard_negatives/synthetic',
        '--force',
    ])
    run_checked(['tar', '-czf', str(drive_prepared), 'datasets/leaf_yolo'])
    print(f'Prepared dataset saved to Drive: {drive_prepared}')
elif local_archive.exists():
    print(f'Preparing from local archive: {local_archive}')
    run_checked([
        'python', 'scripts/prepare_leaf_dataset.py',
        '--archive', str(local_archive),
        '--work-dir', '.',
        '--extra-yolo-dir', 'public/plantdoc_yolo',
        '--hard-negatives-dir', 'hard_negatives/synthetic',
        '--force',
    ])
    run_checked(['tar', '-czf', str(drive_prepared), 'datasets/leaf_yolo'])
    print(f'Prepared dataset saved to Drive: {drive_prepared}')
elif drive_raw_dataset.exists() and allow_slow_drive_folder:
    print(f'Preparing from unzipped Drive dataset: {drive_raw_dataset}')
    print('Warning: this is the slow fallback path. Prefer leaf_yolo_dataset.tar.gz or archive.zip.')
    raw_images = count_files(drive_raw_dataset / 'images', {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
    raw_labels = count_files(drive_raw_dataset / 'labels', {'.txt'})
    print(f'Drive raw image files: {raw_images:,}')
    print(f'Drive raw label files: {raw_labels:,}')
    if raw_images == 0 or raw_labels == 0:
        raise FileNotFoundError(f'Drive raw dataset is missing images or labels under {drive_raw_dataset}')
    if raw_images != raw_labels:
        print('Warning: image/label counts differ. Prep will show exact missing examples if pairing fails.')
    run_checked([
        'python', 'scripts/prepare_leaf_dataset.py',
        '--work-dir', '.',
        '--raw-dir', str(drive_raw_root),
        '--no-extract',
        '--extra-yolo-dir', 'public/plantdoc_yolo',
        '--hard-negatives-dir', 'hard_negatives/synthetic',
        '--copy-images',
        '--force',
    ])
    run_checked(['tar', '-czf', str(drive_prepared), 'datasets/leaf_yolo'])
    print(f'Prepared dataset saved to Drive: {drive_prepared}')
else:
    raise FileNotFoundError('No fast dataset source found. Upload leaf_yolo_dataset.tar.gz to MyDrive/leaf-object-detection/datasets/ or upload archive.zip in the previous cell.')

!python scripts/validate_leaf_dataset.py \
  --dataset datasets/leaf_yolo \
  --write-report

!cp datasets/leaf_yolo/validation_report.json "$DRIVE_ROOT/datasets/validation_report.json"

## 6. Smoke Test

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --smoke \
  --export \
  --drive-out "$DRIVE_ROOT/runs/leaf_yolo"

## 7. Main Accuracy-Focused Fine-Tuning

This trains each pretrained YOLO26 candidate, then runs a lower-augmentation fine-tune stage from that model's `best.pt`. Outputs are synced to Drive after every model.

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --models yolo26s.pt yolo26m.pt yolo26x.pt \
  --epochs 180 \
  --imgsz 640 \
  --fine-tune \
  --finetune-epochs 40 \
  --export \
  --drive-out "$DRIVE_ROOT/runs/leaf_yolo"

## 8. Save Final Artifacts to Drive

In [ ]:
!zip -r "$DRIVE_ROOT/artifacts/leaf_yolo_runs.zip" runs/leaf_yolo
!cp runs/leaf_yolo/training_summary.json "$DRIVE_ROOT/artifacts/training_summary.json" || true
!find "$DRIVE_ROOT/runs/leaf_yolo" -maxdepth 3 \( -name best.pt -o -name last.pt -o -name '*.onnx' \) -print
!echo "All saved under: $DRIVE_ROOT"